# Experimentos de OpenVLA sobre LIBERO

Corre OpenVLA en varias tareas de LIBERO y muestra los resultados (tabla, tasa
de éxito y videos). El código pesado vive en el framework (`core`, `benchmarks`,
`models`); este notebook solo hace **configuración + ejecución + visualización**.

> Requiere un entorno con **GPU** y OpenVLA + LIBERO instalados (jupyter en el
> cluster, o Kaggle). Los pesos deben poder descargarse/estar en `HF_HOME`.

## 1. Setup

In [ ]:
import os, sys, glob

os.environ.setdefault("MUJOCO_GL", "egl")     # render headless (sin pantalla)


def _find_project_root():
    """Sube desde el cwd buscando la raiz del proyecto (simulation.py + core/)."""
    d = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, "simulation.py")) and \
           os.path.isdir(os.path.join(d, "core")):
            return d
        d = os.path.dirname(d)
    hits = glob.glob(os.path.join(os.getcwd(), "**", "simulation.py"), recursive=True)
    if hits:
        return os.path.dirname(os.path.abspath(hits[0]))
    raise RuntimeError("No encontre la raiz del proyecto (simulation.py + core/).")


ROOT = _find_project_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", os.path.join(ROOT, "hf_cache"))   # caché de pesos
print("Proyecto:", ROOT)
print("HF_HOME :", os.environ["HF_HOME"])

## 2. Configuración

In [ ]:
from dataclasses import dataclass


@dataclass
class Config:
    suite: str = "libero_10"
    num_tasks: int = 3             # cuantos escenarios (tareas) probar
    episodes_per_task: int = 1     # configuraciones iniciales por tarea
    max_steps: int = 520           # libero_10 es de horizonte largo (oficial ~520)
    load_in_4bit: bool = True      # 4-bit cabe en una GPU de 11 GB
    out_dir: str = "output/experiments"


cfg = Config()
cfg

## 3. Cargar el modelo (una sola vez)

In [ ]:
from core import View
from models.OpenVLA import OpenVLAController

model = OpenVLAController(view=View.AGENT, center_crop=True,
                         load_in_4bit=cfg.load_in_4bit, device="cuda")
print("modelo cargado | unnorm_key =", model.unnorm_key)

## 4. Correr los experimentos
Un `LiberoController` por tarea (escenario); `run_experiments` corre los
episodios y graba un video por cada uno.

In [ ]:
from core import run_experiments
from benchmarks.libero import LiberoController

task_list = LiberoController.tasks(cfg.suite)[:cfg.num_tasks]
benchmarks = [LiberoController(task_id=tid, suite=cfg.suite) for tid, _ in task_list]

results = run_experiments(
    model, benchmarks,
    max_steps=cfg.max_steps,
    episodes_per_task=cfg.episodes_per_task,
    out_dir=cfg.out_dir,
    record_view=View.AGENT,
)
print(f"{len(results)} episodios corridos")

## 5. Resultados — tabla y tasa de éxito

In [ ]:
import pandas as pd
from core import summarize

df = pd.DataFrame(results)
resumen = summarize(results)
print("Tasa de exito global: {exitos}/{total} = {tasa_exito:.0%}".format(**resumen))
df[["scenario", "instruction", "episode", "success", "steps"]]

In [ ]:
import matplotlib.pyplot as plt

por_escenario = df.groupby("scenario")["success"].mean()
ax = por_escenario.plot(kind="bar", ylim=(0, 1), color="#4C78A8", rot=0)
ax.set_xlabel("escenario"); ax.set_ylabel("tasa de exito")
ax.set_title("Exito por tarea"); plt.tight_layout(); plt.show()

## 6. Resultados — videos

In [ ]:
from IPython.display import Video, display

for r in results:
    if r["video"]:
        print(f"escenario {r['scenario']} | {r['instruction']} | exito={r['success']}")
        display(Video(r["video"], embed=True, width=320))